# Assignment 01: MLP from Scratch

**Module:** 04 — Neural Networks Deep Dive  
**Estimated Time:** 8-12 hours

---

## Learning Objectives

- Build a fully functional MLP from scratch in NumPy — no autograd
- Implement forward pass, backward pass (manual chain rule), and SGD parameter updates
- Implement softmax + cross-entropy loss and derive the gradient simplification
- Train to >95% accuracy on MNIST
- Verify all gradients match PyTorch autograd

### Key Equations

**Forward pass (layer $i$):**
$$\mathbf{z}_i = \mathbf{W}_i \mathbf{a}_{i-1} + \mathbf{b}_i, \quad \mathbf{a}_i = \phi(\mathbf{z}_i)$$

**Softmax + cross-entropy gradient (output layer):**
$$\frac{\partial L}{\partial \mathbf{z}_L} = \hat{\mathbf{y}} - \mathbf{y}_{true}$$

**Backward pass (layer $i$):**
$$\frac{\partial L}{\partial \mathbf{W}_i} = \frac{1}{m} \frac{\partial L}{\partial \mathbf{z}_i} \mathbf{a}_{i-1}^T, \quad
\frac{\partial L}{\partial \mathbf{b}_i} = \frac{1}{m} \sum_j \frac{\partial L}{\partial \mathbf{z}_i^{(j)}}$$
$$\frac{\partial L}{\partial \mathbf{a}_{i-1}} = \mathbf{W}_i^T \frac{\partial L}{\partial \mathbf{z}_i}, \quad
\frac{\partial L}{\partial \mathbf{z}_{i-1}} = \frac{\partial L}{\partial \mathbf{a}_{i-1}} \odot \phi'(\mathbf{z}_{i-1})$$

In [ ]:
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import fetch_openml

sys.path.insert(0, '../../')
from shared_utils.common import set_seed

set_seed(42)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('Setup complete.')

## Helper Functions

In [ ]:
def one_hot(y: np.ndarray, n_classes: int) -> np.ndarray:
    """One-hot encode labels.
    
    Args:
        y: integer labels, shape (n_samples,)
        n_classes: number of classes
    Returns:
        One-hot matrix, shape (n_classes, n_samples)
    """
    oh = np.zeros((n_classes, len(y)))
    oh[y, np.arange(len(y))] = 1
    return oh


def accuracy(y_pred: np.ndarray, y_true: np.ndarray) -> float:
    """Compute classification accuracy.
    
    Args:
        y_pred: predicted probabilities, shape (n_classes, n_samples)
        y_true: one-hot labels, shape (n_classes, n_samples)
    Returns:
        accuracy as a float in [0, 1]
    """
    return np.mean(np.argmax(y_pred, axis=0) == np.argmax(y_true, axis=0))


def plot_training_curves(train_losses, val_accs, train_accs=None):
    """Plot training loss and accuracy curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(train_losses)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Cross-Entropy Loss')
    ax1.set_title('Training Loss')
    
    if train_accs:
        ax2.plot(train_accs, label='Train')
    ax2.plot(val_accs, label='Validation')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Accuracy')
    ax2.legend()
    plt.tight_layout()
    plt.show()

---
## Part 1: NumPy MLP Implementation

Implement the full MLP class. Note: batch dimension is the **second** axis (columns), matching mathematical convention.

In [ ]:
class MLP:
    """Multi-layer perceptron implemented from scratch in NumPy."""

    def __init__(self, layer_sizes: list[int], activation: str = 'relu'):
        """Initialize MLP.

        Args:
            layer_sizes: e.g. [784, 128, 64, 10]
            activation: 'relu', 'sigmoid', or 'tanh'
        """
        self.num_layers = len(layer_sizes) - 1
        self.params = {}
        self.activation_name = activation

        # Initialize weights (He for ReLU, Xavier for others)
        for i in range(self.num_layers):
            n_in = layer_sizes[i]
            n_out = layer_sizes[i + 1]
            if activation == 'relu':
                scale = np.sqrt(2.0 / n_in)
            else:
                scale = np.sqrt(2.0 / (n_in + n_out))
            self.params[f'W{i}'] = np.random.randn(n_out, n_in) * scale
            self.params[f'b{i}'] = np.zeros((n_out, 1))

    def activation(self, z: np.ndarray) -> np.ndarray:
        """Apply activation function."""
        # YOUR CODE HERE
        # Support: 'relu', 'sigmoid', 'tanh'
        pass

    def activation_derivative(self, z: np.ndarray) -> np.ndarray:
        """Compute derivative of activation w.r.t. pre-activation z."""
        # YOUR CODE HERE
        # relu'(z) = 1 if z > 0, else 0
        # sigmoid'(z) = sigmoid(z) * (1 - sigmoid(z))
        # tanh'(z) = 1 - tanh(z)^2
        pass

    def softmax(self, z: np.ndarray) -> np.ndarray:
        """Numerically stable softmax.
        
        Args:
            z: shape (n_classes, batch_size)
        Returns:
            shape (n_classes, batch_size), columns sum to 1
        """
        # YOUR CODE HERE
        # Subtract max per column for stability
        pass

    def forward(self, x: np.ndarray) -> np.ndarray:
        """Forward pass. Store all intermediates in self.cache.

        Args:
            x: shape (n_features, batch_size)
        Returns:
            y_hat: shape (n_classes, batch_size)
        """
        self.cache = {'a-1': x}  # a_{-1} is the input
        a = x

        for i in range(self.num_layers):
            # YOUR CODE HERE
            # z_i = W_i @ a + b_i
            # a_i = activation(z_i) for hidden layers, softmax for last layer
            # Store z_i and a_i in self.cache
            pass

        return a  # final activation = y_hat

    def compute_loss(self, y_hat: np.ndarray, y_true: np.ndarray) -> float:
        """Cross-entropy loss.

        Args:
            y_hat: predicted probs, shape (n_classes, batch_size)
            y_true: one-hot labels, shape (n_classes, batch_size)
        Returns:
            scalar loss
        """
        # YOUR CODE HERE
        # L = -(1/m) * sum(y_true * log(y_hat + epsilon))
        pass

    def backward(self, y_hat: np.ndarray, y_true: np.ndarray) -> dict:
        """Backward pass. Compute gradients for all parameters.

        Args:
            y_hat: predicted probs, shape (n_classes, batch_size)
            y_true: one-hot labels, shape (n_classes, batch_size)
        Returns:
            grads: dict with keys dW0, db0, dW1, db1, ...
        """
        grads = {}
        m = y_true.shape[1]

        # Output layer: dz = y_hat - y_true (softmax + CE simplification)
        # YOUR CODE HERE
        # For each layer i from last to first:
        #   dW_i = (1/m) * dz_i @ a_{i-1}.T
        #   db_i = (1/m) * sum(dz_i, axis=1, keepdims=True)
        #   da_{i-1} = W_i.T @ dz_i
        #   dz_{i-1} = da_{i-1} * activation_derivative(z_{i-1})
        pass

        return grads

    def update_params(self, grads: dict, learning_rate: float):
        """Vanilla SGD update."""
        for i in range(self.num_layers):
            self.params[f'W{i}'] -= learning_rate * grads[f'dW{i}']
            self.params[f'b{i}'] -= learning_rate * grads[f'db{i}']

    def train_step(self, x: np.ndarray, y: np.ndarray,
                   learning_rate: float) -> float:
        """One training step: forward -> loss -> backward -> update."""
        y_hat = self.forward(x)
        loss = self.compute_loss(y_hat, y)
        grads = self.backward(y_hat, y)
        self.update_params(grads, learning_rate)
        return loss

In [ ]:
# --- Sanity test on small data ---
np.random.seed(42)
mlp_test = MLP([4, 3, 2], activation='relu')
x_test = np.random.randn(4, 5)  # 4 features, 5 samples
y_test = one_hot(np.array([0, 1, 0, 1, 0]), 2)

y_hat = mlp_test.forward(x_test)
assert y_hat.shape == (2, 5), f'Expected (2,5), got {y_hat.shape}'
assert np.allclose(y_hat.sum(axis=0), 1.0), 'Softmax outputs should sum to 1'

loss = mlp_test.compute_loss(y_hat, y_test)
assert loss > 0, 'Loss should be positive'

grads = mlp_test.backward(y_hat, y_test)
assert 'dW0' in grads and 'dW1' in grads, 'Gradients should have dW0, dW1'
assert grads['dW0'].shape == mlp_test.params['W0'].shape, 'Gradient shape mismatch'
print('Sanity test passed!')

---
## Training on MNIST

Load MNIST, preprocess, and train. Target: **>95% test accuracy**.

In [ ]:
# --- Load and preprocess MNIST ---
print('Loading MNIST...')
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X_all = mnist.data.astype(np.float64) / 255.0  # Normalize to [0, 1]
y_all = mnist.target.astype(int)

# Split: train (55k), val (5k), test (10k)
X_train = X_all[:55000].T   # (784, 55000)
y_train_labels = y_all[:55000]
y_train = one_hot(y_train_labels, 10)

X_val = X_all[55000:60000].T
y_val_labels = y_all[55000:60000]
y_val = one_hot(y_val_labels, 10)

X_test = X_all[60000:].T
y_test_labels = y_all[60000:]
y_test = one_hot(y_test_labels, 10)

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

In [ ]:
# --- Training loop ---
# YOUR CODE HERE
# 1. Create MLP([784, 128, 64, 10], activation='relu')
# 2. Train for 50+ epochs with mini-batch SGD (batch_size=128)
# 3. Track training loss, training accuracy, validation accuracy
# 4. Plot training curves
# 5. Report final test accuracy (target: >95%)
pass

---
## Part 2: PyTorch Verification

### Gradient Matching

In [ ]:
class PyTorchMLP(nn.Module):
    """PyTorch equivalent of NumPy MLP for gradient verification."""

    def __init__(self, layer_sizes: list[int], activation: str = 'relu'):
        super().__init__()
        layers = []
        for i in range(len(layer_sizes) - 1):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
            if i < len(layer_sizes) - 2:
                if activation == 'relu':
                    layers.append(nn.ReLU())
                elif activation == 'sigmoid':
                    layers.append(nn.Sigmoid())
                elif activation == 'tanh':
                    layers.append(nn.Tanh())
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [ ]:
def copy_weights(numpy_mlp: MLP, pytorch_mlp: PyTorchMLP):
    """Copy weights from NumPy MLP to PyTorch MLP.
    
    Note: nn.Linear stores weights as (out, in), matching our convention.
    But nn.Linear computes output = input @ W.T + b, so input is (batch, features).
    Our NumPy MLP uses (features, batch) convention.
    """
    # YOUR CODE HERE
    pass


def compare_gradients(numpy_grads: dict, pytorch_model: PyTorchMLP):
    """Compare NumPy gradients to PyTorch gradients."""
    # YOUR CODE HERE
    # For each parameter, compute max absolute difference
    # Should be < 1e-5
    pass

In [ ]:
# --- Gradient matching test ---
# YOUR CODE HERE
# 1. Create both models with same architecture
# 2. Copy weights from NumPy to PyTorch
# 3. Run one forward+backward on same batch
# 4. Compare gradients (max diff should be < 1e-5)
pass

### Training Comparison

In [ ]:
# --- Training comparison ---
# YOUR CODE HERE
# Train both models on MNIST with same hyperparameters
# Plot training loss curves on the same graph
# Report final test accuracy for both
pass

---
## Part 3: Analysis

### Gradient Derivation

Derive $\frac{\partial L}{\partial \mathbf{W}_0}$ for the first layer. Show every step of the chain rule.

*YOUR DERIVATION HERE*

### Softmax-Cross-Entropy Simplification

Derive why $\frac{\partial L}{\partial \mathbf{z}_L} = \hat{\mathbf{y}} - \mathbf{y}_{true}$ when combining softmax and cross-entropy.

*YOUR DERIVATION HERE*

### Numerical Gradient Check

For a small network [4, 3, 2], verify gradients numerically.

In [ ]:
# --- Numerical gradient check ---
# YOUR CODE HERE
# For each parameter W_ij:
#   dL/dW_ij ~ (L(W_ij + eps) - L(W_ij - eps)) / (2 * eps)
# Compare to analytical gradients
# Report max relative error
pass

### What I Learned

*YOUR REFLECTION HERE*